In [9]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

In [58]:
class ReTanh(nn.Module):
        
    def forward(self, x):
        return torch.maximum(torch.zeros_like(x), torch.tanh(x))

class MetaRNN(nn.Module):
    def __init__(self, 
                 n_inputs, n_neurons, n_outputs, 
                 n_meta_inputs, n_meta_hidden_layers, n_meta_hidden_units,
                 activation_func=ReTanh, **kwargs):

        self.n_neurons = n_neurons
        self.n_inputs = n_inputs
        self.n_outputs = n_outputs

        self.n_meta_inputs = n_meta_inputs
        self.n_meta_hidden_layers = n_meta_hidden_layers
        self.n_meta_hidden_units = n_meta_hidden_units

        self.dt = kwargs.get('dt', 0.1)
        self.tau = kwargs.get('tau', 1)
        self.hidden_g = kwargs.get('hidden_g', 1.1)

        self.learn_x_0 = kwargs.get('learn_x_0', True)
        self.learn_W_in = kwargs.get('learn_W_in', True)
        self.learn_W_out = kwargs.get('learn_W_out', True)

        self.state_noise_std = kwargs.get('state_noise_std', 0.1)

        self.solver = kwargs.get('solver', 'euler')

        self.device = kwargs.get('device', 'cpu')

        super(MetaRNN, self).__init__()

        self.activation_func = activation_func()

        self.W_in = nn.Linear(self.n_inputs, self.n_neurons, bias=True)
        nn.init.normal_(self.W_in.weight, mean=0, std=self.hidden_g / np.sqrt(self.n_inputs))
        self.W_in.weight.requires_grad = self.learn_W_in

        input_bias = 0.1 + 0.01*torch.randn(self.n_neurons)
        self.W_in.bias = torch.nn.Parameter(torch.squeeze(input_bias))
        self.W_in.bias.requires_grad = self.learn_W_in or self.learn_W_in_norm
    
        self.W_out = nn.Linear(n_neurons, self.n_outputs, bias=True)
        self.W_out.weight.requires_grad = self.learn_W_out
        output_bias = 0.1 + 0.01*torch.randn(self.n_outputs)
        self.W_out.bias = torch.nn.Parameter(torch.squeeze(output_bias))
        self.W_out.bias.requires_grad = self.learn_W_out or self.learn_W_out_norm

        layers = [
            nn.Sequential(
                nn.Linear(in_features=n_meta_inputs, out_features=n_meta_hidden_units), self.activation_func
            ),
            nn.Sequential(
                nn.Linear(in_features=n_meta_hidden_units, out_features=n_neurons**2)
            )
        ]
        for _ in range(n_meta_hidden_layers-1):
            layers.insert(1, nn.Sequential(
                nn.Linear(in_features=n_meta_hidden_units, out_features=n_meta_hidden_units), self.activation_func
            ))
        self.meta_network = torch.nn.Sequential(*layers)

        self.x_0 = torch.nn.Parameter(torch.zeros(self.n_neurons), requires_grad=self.learn_x_0)

        self.to(self.device)

    def forward(self, meta_u: torch.Tensor, u: torch.Tensor):
        n_trials, n_timesteps, _ = u.shape
        assert u.shape[2] == self.n_inputs
        assert meta_u.shape[0] == n_trials and meta_u.shape[1] == self.n_meta_inputs

        W_rec = self.meta_network(meta_u).reshape(n_trials, self.n_neurons, self.n_neurons)

        u = u.transpose(0, 1)

        X = [self.x_0.reshape((1,self.n_neurons)).repeat((n_trials,1))]
        Z = []

        def F(x, r, u, noise):
            x_step = -x + torch.einsum('bij,bj->bi', W_rec, r) + self.W_in(u) + noise
            return (1/self.tau) * x_step

        for t in range(n_timesteps):
            x_t = X[-1]
            r_t = self.activation_func(x_t)
            u_t = u[t] 
            state_noise_t = torch.normal(mean=0, std=self.state_noise_std, size=(n_trials, self.n_neurons), device=self.device)

            # Continuous-Time RNN Update Funcion:
            if self.solver == 'euler':
                x_next = x_t + self.dt * F(x_t, r_t, u_t, state_noise_t)
                
            elif self.solver == 'rk4':
                x_next = x_t

                k1 = F(x_t, r_t, u_t, state_noise_t)
                x_next += (self.dt/6) * k1

                k2 = F(x_t + 0.5 * self.dt * k1, r_t, u_t, state_noise_t)
                x_next += (self.dt/3) * k2
                del k1

                k3 = F(x_t + 0.5 * self.dt * k2, r_t, u_t, state_noise_t)
                x_next += (self.dt/3) * k3
                del k2

                k4 = F(x_t + self.dt * k3, r_t, u_t, state_noise_t)
                x_next += (self.dt/6) * k4
                del k3, k4

            else:
                raise ValueError(f'Unsupported solver: {self.solver}')
            
            z_next = self.W_out(self.activation_func(x_next))

            X.append(x_next)
            Z.append(z_next)

        return torch.stack(X[1:], dim=1), torch.stack(Z, dim=1), W_rec.reshape((n_trials, -1))


        

In [16]:
def get_vars(batch_size, n_timesteps, init_duration=10, av_step_std=0.03, av_step_momentum=0.5, av_step_zero_prob=0.5):
    
    # Randomly select starting angle for each sequence
    angle_0 = (torch.rand(batch_size)) * 2 * np.pi

    # Initialise tensors to store the target angle and input angular velocity for each sequence
    angle, angular_velocity = torch.zeros((batch_size, n_timesteps)), torch.zeros((batch_size, n_timesteps))

    zero_trials = torch.where(torch.rand((batch_size,)) < av_step_zero_prob)

    normal = torch.distributions.normal.Normal(loc=torch.zeros((batch_size,)), scale=torch.ones((batch_size,))*av_step_std)
    for t in range(init_duration, n_timesteps):
        av_step = normal.sample() + av_step_momentum * angular_velocity[:, t-1]

        if t > n_timesteps*(1/4) and t < n_timesteps*(3/4):
            av_step[zero_trials] = 0

        angular_velocity[:,t] = av_step
    
    # Compute sequence's target angle as its initial angle + integral of angular velocity up to each timestep
    angle = torch.tile(angle_0.reshape((batch_size,1)), dims=(1,n_timesteps)) + torch.cumsum(angular_velocity, dim=1)
    angle = torch.remainder(angle, 2*np.pi)

    # Initialise allocentric target angle (relative to zero head-direction) for each sequence
    allo_shelter_angle_0 = (torch.rand(batch_size) - 1) * 2 * np.pi
    # Create time-varying allocentric angle as difference between constant allocentric target and
    # current head direction
    ego_sheler_angle = allo_shelter_angle_0.reshape((batch_size,1)).repeat((1,n_timesteps)) - angle
    ego_sheler_angle = torch.remainder(ego_sheler_angle, 2 * np.pi)

    return {'av': angular_velocity, 
            'hd': angle, 
            'sd': ego_sheler_angle, 
            'sx': torch.cos(allo_shelter_angle_0), 
            'sy': torch.sin(allo_shelter_angle_0)}

In [64]:
n_epochs = 1000
batch_size = 100
n_timesteps = 250
lambda_rates = 0.01
lambda_weights = 0.01

net = MetaRNN(n_inputs=3, n_neurons=100, n_outputs=2, n_meta_inputs=2, n_meta_hidden_layers=2, n_meta_hidden_units=256)
optim = torch.optim.Adam(net.parameters(), lr=1e-4)

In [65]:
for i in range(n_epochs):
    optim.zero_grad()

    meta_inputs = torch.zeros((batch_size, 2))
    inputs = torch.zeros((batch_size, n_timesteps, 3))
    targets = torch.zeros((batch_size, n_timesteps, 2))
    mask = torch.zeros_like(targets).type(torch.bool)

    vars = get_vars(batch_size=batch_size, n_timesteps=n_timesteps)

    meta_inputs[:,0] = vars['sx']
    meta_inputs[:,1] = vars['sy']
        
    inputs[:,:,0] = vars['av']
    inputs[:,:10,1] = torch.sin(vars['hd'][:,0]).reshape((batch_size,1)).repeat((1,10))
    inputs[:,:10,2] = torch.cos(vars['hd'][:,0]).reshape((batch_size,1)).repeat((1,10))

    targets[:,:,0] = torch.sin(vars['sd'])
    targets[:,:,1] = torch.cos(vars['sd'])

    states, outputs, W_rec = net(meta_inputs, inputs)
    
    loss = torch.square(outputs - targets)[:,10:].mean() + lambda_rates*(net.activation_func(states).mean()) + lambda_weights*(W_rec.mean())

    loss.backward()
    optim.step()
    
    print(f'[{i}] {loss.item()}')
    

[0] 0.5029758810997009
[1] 0.502602756023407
[2] 0.4994257688522339
[3] 0.4876098334789276
[4] 0.49770283699035645
[5] 0.4823528826236725
[6] 0.48114705085754395
[7] 0.47221672534942627
[8] 0.47530069947242737
[9] 0.4481377899646759
[10] 0.44605714082717896
[11] 0.43955329060554504
[12] 0.4205482602119446
[13] 0.4424003064632416
[14] 0.425712913274765
[15] 0.3809893727302551
[16] 0.37549886107444763
[17] 0.3806780278682709
[18] 0.3669780194759369
[19] 0.37235966324806213
[20] 0.3224899172782898
[21] 0.33831626176834106
[22] 0.32490572333335876
[23] 0.3422386348247528
[24] 0.3259718716144562
[25] 0.3248511850833893
[26] 0.30584612488746643
[27] 0.30342531204223633
[28] 0.3147815465927124
[29] 0.3010149598121643
[30] 0.315921813249588
[31] 0.3417665362358093
[32] 0.3486492931842804
[33] 0.3454241454601288
[34] 0.28604280948638916
[35] 0.29330119490623474
[36] 0.31073781847953796
[37] 0.3128432035446167
[38] 0.3011642396450043
[39] 0.3148995637893677
[40] 0.30428794026374817
[41] 0.327513

In [2]:
import torch
import torch.nn as nn
from torch.distributions import MultivariateNormal
import numpy as np


class PointwiseMLP(nn.Module):
    """
    Maps (B, K, D) -> (B, K, H) with no interaction across B or K.
    Configurable depth/width.
    """
    def __init__(
        self,
        input_dim: int,
        output_dim: int,
        width: int = 128,
        depth: int = 3,
        activation: nn.Module = nn.SiLU(),
        layer_norm: bool = False,
        bias: bool = True,
    ):
        super().__init__()
        assert depth >= 1, "depth must be >= 1"

        layers = []
        in_dim = input_dim

        if depth == 1:
            layers.append(nn.Linear(in_dim, output_dim, bias=bias))
        else:
            # first hidden
            layers.append(nn.Linear(in_dim, width, bias=bias))
            if layer_norm:
                layers.append(nn.LayerNorm(width))
            layers.append(activation)

            # middle hidden(s)
            for _ in range(depth - 2):
                layers.append(nn.Linear(width, width, bias=bias))
                if layer_norm:
                    layers.append(nn.LayerNorm(width))
                layers.append(activation)

            # output
            layers.append(nn.Linear(width, output_dim, bias=bias))

        self.net = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x: (B, K, D)
        returns: (B, K, H)
        """
        if x.ndim != 3:
            raise ValueError(f"Expected x.ndim==3 (B,K,D), got shape {tuple(x.shape)}")
        B, K, D = x.shape
        y = self.net(x.reshape(B * K, D))
        return y.reshape(B, K, -1)


def sample_neural_kernel_grid(
    N: int,
    B: int,
    net: nn.Module,
    *,
    jitter: float = 1e-5,
    normalize_inner_product: bool = True,
) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Returns:
      samples: (B, N, N) where each sample is drawn from N(0, Sigma)
      Sigma:   (N^2, N^2) covariance matrix, Sigma_ij = <h(x_i), h(x_j)>

    Construction:
      - Create N^2 points on [0,1]x[0,1] on an evenly-spaced grid (D=2)
      - Compute hidden reps h(x) = net(x) in R^H
      - Sigma = H H^T (optionally /H), add jitter*I
      - Draw B samples using rsample() (reparameterized, differentiable)
    """
    if N <= 0 or B <= 0:
        raise ValueError("N and B must be positive integers")

    # Infer device/dtype from network parameters
    try:
        p = next(net.parameters())
        device, dtype = p.device, p.dtype
    except StopIteration:
        device, dtype = torch.device("cpu"), torch.float32

    # (N^2, 2) grid in [0,1]^2
    t = torch.linspace(0.0, 1.0, steps=N, device=device, dtype=dtype)
    yy, xx = torch.meshgrid(t, t, indexing="ij")
    grid = torch.stack([xx, yy], dim=-1).reshape(N * N, 2)  # (K,2), K=N^2

    # Hidden reps for each grid point: (1, K, H) -> (K, H)
    h = net(grid.unsqueeze(0))[0]  # (K, H)
    K, H = h.shape

    # Covariance: (K, K)
    Sigma = h @ h.transpose(0, 1)
    if normalize_inner_product:
        Sigma = Sigma / max(H, 1)

    # Jitter for numerical stability (ensures PD for Cholesky-based sampling)
    Sigma = (1/N) * Sigma + (jitter * torch.eye(K, device=device, dtype=dtype))

    # Reparameterized sampling: (B, K)
    mvn = MultivariateNormal(loc=torch.zeros(K, device=device, dtype=dtype),
                             covariance_matrix=Sigma)
    w = mvn.rsample((B,))  # differentiable w.r.t. Sigma -> h -> net params

    samples = w.reshape(B, N, N)
    return samples, Sigma


In [3]:
class ReTanh(nn.Module):
        
    def forward(self, x):
        return torch.maximum(torch.zeros_like(x), torch.tanh(x))

class GPRNN(nn.Module):
    def __init__(self, 
                 n_inputs, n_neurons, n_outputs,
                 mlp_input_dim, mlp_output_dim, mlp_depth=3, mlp_width=128,
                 activation_func=ReTanh, **kwargs):

        self.n_neurons = n_neurons
        self.n_inputs = n_inputs
        self.n_outputs = n_outputs

        self.dt = kwargs.get('dt', 0.1)
        self.tau = kwargs.get('tau', 1)
        self.hidden_g = kwargs.get('hidden_g', 1.1)

        self.learn_x_0 = kwargs.get('learn_x_0', True)
        self.learn_W_in = kwargs.get('learn_W_in', True)
        self.learn_W_out = kwargs.get('learn_W_out', True)

        self.state_noise_std = kwargs.get('state_noise_std', 0.1)
        self.solver = kwargs.get('solver', 'euler')
        self.device = kwargs.get('device', 'cpu')

        super(GPRNN, self).__init__()

        self.kernel_mlp = PointwiseMLP(mlp_input_dim, mlp_output_dim, mlp_width, mlp_depth)

        self.activation_func = activation_func()

        self.W_in = nn.Linear(self.n_inputs, self.n_neurons, bias=True)
        nn.init.normal_(self.W_in.weight, mean=0, std=self.hidden_g / np.sqrt(self.n_inputs))
        self.W_in.weight.requires_grad = self.learn_W_in
        input_bias = 0.1 + 0.01*torch.randn(self.n_neurons)
        self.W_in.bias = torch.nn.Parameter(torch.squeeze(input_bias))
        self.W_in.bias.requires_grad = self.learn_W_in
    
        self.W_out = nn.Linear(n_neurons, self.n_outputs, bias=True)
        self.W_out.weight.requires_grad = self.learn_W_out
        output_bias = 0.1 + 0.01*torch.randn(self.n_outputs)
        self.W_out.bias = torch.nn.Parameter(torch.squeeze(output_bias))
        self.W_out.bias.requires_grad = self.learn_W_out 

        self.x_0 = torch.nn.Parameter(torch.zeros(self.n_neurons), requires_grad=self.learn_x_0)

        self.to(self.device)

    def forward(self, u: torch.Tensor):
        n_trials, n_timesteps, _ = u.shape
        assert u.shape[2] == self.n_inputs
        u = u.transpose(0, 1)

        print('sampling recurrency...')
        W_rec, cov = sample_neural_kernel_grid(N=self.n_neurons, B=n_trials, net=self.kernel_mlp)

        L = torch.linalg.cholesky(cov)  # supports batching and is differentiable
        diag = torch.diagonal(L, dim1=-2, dim2=-1)  # (..., K)
        half_logdet = torch.sum(torch.log(diag), dim=-1)  # (...,) == 0.5*logdet

        x_t = self.x_0.reshape((1,self.n_neurons)).repeat((n_trials,1))
        Z = []

        def F(x, r, u, noise):
            x_step = -x + torch.einsum('bij,bj->bi', W_rec, r) + self.W_in(u) + noise
            return (1/self.tau) * x_step

        print('computing forward pass...')
        for t in range(n_timesteps):
            r_t = self.activation_func(x_t)
            u_t = u[t] 
            state_noise_t = torch.normal(mean=0, std=self.state_noise_std, size=(n_trials, self.n_neurons), device=self.device)

            # Continuous-Time RNN Update Funcion:
            if self.solver == 'euler':
                x_next = x_t + self.dt * F(x_t, r_t, u_t, state_noise_t)
                
            elif self.solver == 'rk4':
                x_next = x_t

                k1 = F(x_t, r_t, u_t, state_noise_t)
                x_next += (self.dt/6) * k1

                k2 = F(x_t + 0.5 * self.dt * k1, r_t, u_t, state_noise_t)
                x_next += (self.dt/3) * k2
                del k1

                k3 = F(x_t + 0.5 * self.dt * k2, r_t, u_t, state_noise_t)
                x_next += (self.dt/3) * k3
                del k2

                k4 = F(x_t + self.dt * k3, r_t, u_t, state_noise_t)
                x_next += (self.dt/6) * k4
                del k3, k4

            else:
                raise ValueError(f'Unsupported solver: {self.solver}')
            
            z_next = self.W_out(self.activation_func(x_next))

            x_t = x_next
            Z.append(z_next)

        return torch.stack(Z, dim=1), -half_logdet


        

In [4]:
def get_vars(batch_size, n_timesteps, init_duration=10, av_step_std=0.03, av_step_momentum=0.5, av_step_zero_prob=0.5):
    
    # Randomly select starting angle for each sequence
    angle_0 = (torch.rand(batch_size)) * 2 * np.pi

    # Initialise tensors to store the target angle and input angular velocity for each sequence
    angle, angular_velocity = torch.zeros((batch_size, n_timesteps)), torch.zeros((batch_size, n_timesteps))

    zero_trials = torch.where(torch.rand((batch_size,)) < av_step_zero_prob)

    normal = torch.distributions.normal.Normal(loc=torch.zeros((batch_size,)), scale=torch.ones((batch_size,))*av_step_std)
    for t in range(init_duration, n_timesteps):
        av_step = normal.sample() + av_step_momentum * angular_velocity[:, t-1]

        if t > n_timesteps*(1/4) and t < n_timesteps*(3/4):
            av_step[zero_trials] = 0

        angular_velocity[:,t] = av_step
    
    # Compute sequence's target angle as its initial angle + integral of angular velocity up to each timestep
    angle = torch.tile(angle_0.reshape((batch_size,1)), dims=(1,n_timesteps)) + torch.cumsum(angular_velocity, dim=1)
    angle = torch.remainder(angle, 2*np.pi)

    # Initialise allocentric target angle (relative to zero head-direction) for each sequence
    allo_shelter_angle_0 = (torch.rand(batch_size) - 1) * 2 * np.pi
    # Create time-varying allocentric angle as difference between constant allocentric target and
    # current head direction
    ego_sheler_angle = allo_shelter_angle_0.reshape((batch_size,1)).repeat((1,n_timesteps)) - angle
    ego_sheler_angle = torch.remainder(ego_sheler_angle, 2 * np.pi)

    return {'av': angular_velocity, 
            'hd': angle, 
            'sd': ego_sheler_angle, 
            'sx': torch.cos(allo_shelter_angle_0), 
            'sy': torch.sin(allo_shelter_angle_0)}

In [5]:
n_epochs = 1000
batch_size = 100
n_timesteps = 100
lambda_entropy = 0.000001

net = GPRNN(n_inputs=5, n_neurons=100, n_outputs=2, 
            mlp_input_dim=2, mlp_output_dim=256, mlp_depth=2,
            learn_W_in=False, learn_W_out=False)
optim = torch.optim.Adam(net.parameters(), lr=1e-4)

In [ ]:
for i in range(n_epochs):
    optim.zero_grad()

    inputs = torch.zeros((batch_size, n_timesteps, 5))
    targets = torch.zeros((batch_size, n_timesteps, 2))

    vars = get_vars(batch_size=batch_size, n_timesteps=n_timesteps)

    inputs[:,:,0] = vars['av']
    inputs[:,:10,1] = torch.sin(vars['hd'][:,0]).reshape((batch_size,1)).repeat((1,10))
    inputs[:,:10,2] = torch.cos(vars['hd'][:,0]).reshape((batch_size,1)).repeat((1,10))
    inputs[:,:10,3] = vars['sx'].reshape((batch_size,1)).repeat((1,10))
    inputs[:,:10,4] = vars['sy'].reshape((batch_size,1)).repeat((1,10))

    targets[:,:,0] = torch.sin(vars['sd'])
    targets[:,:,1] = torch.cos(vars['sd'])

    outputs, entropies = net(inputs)

    mse = torch.square(outputs - targets)[:,10:].mean()
    entropy = entropies.mean()
    
    loss = mse - lambda_entropy * entropy

    print('differentiating...')
    loss.backward()
    optim.step()
    
    print(f'[{i}] {loss.item():.4f} | {mse.item():.4f} | {entropy.item():.4f}')
    

sampling recurrency...
